### Task 5: Model Iterations

Victoria Vicheva (233182)

Team 9

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

1.Load the dataset

In [2]:
# Load dataset
file_paths = ["goemotions_1.csv", "goemotions_2.csv", "goemotions_3.csv"]
dfs = [pd.read_csv(file) for file in file_paths]
df = pd.concat(dfs, ignore_index=True)

In [3]:
# Define core emotion mapping
core_emotions = {
    "happiness": [
        "amusement",
        "approval",
        "excitement",
        "gratitude",
        "joy",
        "love",
        "optimism",
        "pride",
        "relief",
    ],
    "sadness": ["grief", "remorse", "sadness"],
    "surprise": ["realization", "surprise"],
    "anger": ["anger", "annoyance", "disapproval"],
    "disgust": ["disgust"],
    "fear": ["fear", "nervousness"],
    "neutral": ["neutral"],
}

In [4]:
# Function to map emotions to core categories
def map_to_core_emotions(row):
    for core, emotions in core_emotions.items():
        if any(row[emotion] == 1 for emotion in emotions):
            return core
    return "neutral"


df["core_emotion"] = df.apply(map_to_core_emotions, axis=1)

In [5]:
# Select only text and core emotions
df_clean = df[["text", "core_emotion"]].dropna()

# Split data
X = df_clean["text"]
y = df_clean["core_emotion"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
# Convert text to numerical features using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [7]:
# Define SMOTE strategy (only for minority classes)
smote_strategy = {"disgust": 5000, "fear": 5000, "surprise": 12000, "sadness": 10000}

In [8]:
# Apply SMOTE for minority class oversampling
smote = SMOTE(sampling_strategy=smote_strategy, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_tfidf, y_train)

In [9]:
# Define undersampling strategy (only for majority classes)
undersample_strategy = {"neutral": 10000, "happiness": 7000}

In [10]:
# Apply undersampling for majority class reduction
under = RandomUnderSampler(sampling_strategy=undersample_strategy, random_state=42)
X_train_resampled, y_train_resampled = under.fit_resample(
    X_train_resampled, y_train_resampled
)

In [11]:
# Train Naïve Bayes model
nb_model = MultinomialNB()
nb_model.fit(X_train_resampled, y_train_resampled)

MultinomialNB()

In [12]:
# Make predictions
y_pred = nb_model.predict(X_test_tfidf)

In [13]:
# Evaluate model performance
classification_results = classification_report(y_test, y_pred)
print("Classification Report:\n", classification_results)

Classification Report:
               precision    recall  f1-score   support

       anger       0.17      0.85      0.29      5463
     disgust       0.33      0.10      0.15       668
        fear       0.38      0.34      0.36       666
   happiness       0.81      0.26      0.40     12738
     neutral       0.59      0.10      0.16     18789
     sadness       0.31      0.48      0.38      1717
    surprise       0.17      0.37      0.23      2204

    accuracy                           0.28     42245
   macro avg       0.39      0.36      0.28     42245
weighted avg       0.56      0.28      0.27     42245

